# 18 — Non-Cubic Calibrants (Al₂O₃ corundum, custom)

The cubic calibrants (CeO₂, LaB₆, Si) are convenient because every ring is determined by a single `a` lattice constant. Real samples are often trigonal, hexagonal, or orthorhombic — and for high-2θ work, **corundum (Al₂O₃)** is a popular alternative calibrant: trigonal R3̄c (SG 167), `a = 4.7589 Å`, `c = 12.9920 Å`, more rings per unit 2θ, less peak overlap than CeO₂.

The one-shot `calibrate()` entry point already knows about Al₂O₃ via its built-in calibrant database. For truly custom materials, pass a dict.

This notebook covers:

1. The built-in calibrants and how to inspect them.
2. Using the dict form for a non-cubic calibrant.
3. The d-spacing formulas v2 currently knows (cubic, trigonal/hex).
4. Adding your own to the database.

In [ ]:
import os; os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
import math, numpy as np
from midas_calibrate_v2 import calibrate, CALIBRANTS

## 1. Built-in calibrants

`CALIBRANTS` is a public dict of canonical recipes.

In [ ]:
for name, recipe in CALIBRANTS.items():
    print(f'  {name:8s}  a={recipe["a"]} Å'
          + (f'  c={recipe["c"]} Å' if 'c' in recipe else '')
          + (f'  γ={recipe["gamma"]}°' if 'gamma' in recipe else '')
          + f'  SG={recipe["sg"]}')

## 2. Calibrate with Al₂O₃ (trigonal)

Pass `calibrant='Al2O3'`. The `_generate_sim_radii_px` helper picks the **trigonal d-spacing formula** automatically based on `alpha=90, gamma=120`:

$$\frac{1}{d^2} = \frac{4}{3} \frac{h^2 + hk + k^2}{a^2} + \frac{l^2}{c^2}$$

For a real corundum image, the call is identical to CeO₂ — only the `calibrant` string changes.

In [ ]:
# Same one-shot call as CeO₂, just the calibrant name changes:
#
# result = calibrate(image, wavelength=0.184139, pxY=150.0, dark=dark,
#                    calibrant='Al2O3', output_dir='./al2o3_out')
#
# Show that the ring predictions differ from CeO₂.
from midas_calibrate_v2.pipelines.auto import _generate_sim_radii_px

LAM = 0.184139; PX = 150.0; LSD = 840_000.0
for name, recipe in CALIBRANTS.items():
    radii = _generate_sim_radii_px(
        lattice_a=recipe['a'], lattice_c=recipe.get('c', recipe['a']),
        alpha=recipe.get('alpha', 90.0), gamma=recipe.get('gamma', 90.0),
        wavelength=LAM, px=PX, Lsd_nominal_um=LSD, max_2theta_deg=12.0,
    )
    print(f'  {name:8s} -> first 6 rings (R px @ Lsd={LSD/1000} mm): '
          f'{[round(r,1) for r in radii[:6]]}')

## 3. The dict form — custom materials

If you have a calibrant not in the built-in database, pass a dict. Required keys:
* `a` — primary lattice constant (Å)
* `sg` — space group number (used for selection rules in production, not yet applied in `_generate_sim_radii_px`)

Optional:
* `c`, `alpha`, `gamma` — for non-cubic systems

Currently `_generate_sim_radii_px` handles **cubic** (`alpha=90, gamma=90, a=c`) and **trigonal/hexagonal** (`alpha=90, gamma=120`). For other systems, add the appropriate d-spacing branch in `pipelines/auto.py`.

In [ ]:
# Example: custom hexagonal calibrant (e.g., a hexagonal-close-packed metal)
custom = {'a': 2.95, 'c': 4.68, 'alpha': 90.0, 'gamma': 120.0, 'sg': 194}
radii = _generate_sim_radii_px(
    lattice_a=custom['a'], lattice_c=custom['c'],
    alpha=custom['alpha'], gamma=custom['gamma'],
    wavelength=LAM, px=PX, Lsd_nominal_um=LSD,
)
print(f'custom hex (a={custom["a"]}, c={custom["c"]}): first 8 rings: '
      f'{[round(r,1) for r in radii[:8]]}')

# Used in calibrate() like:
# result = calibrate(image, wavelength=LAM, pxY=PX, dark=dark, calibrant=custom)

## 4. Add a calibrant to the global database

If you use a custom calibrant repeatedly, add it to `CALIBRANTS` at startup:

```python
from midas_calibrate_v2 import CALIBRANTS
CALIBRANTS['MyMaterial'] = {'a': 5.123, 'c': 8.456,
                              'alpha': 90.0, 'gamma': 120.0, 'sg': 167}
# Now everywhere accepts calibrant='MyMaterial'
```

For permanent registration, edit `midas_calibrate_v2/pipelines/auto.py:CALIBRANTS`.

## Recommendations

| use case | calibrant |
|---|---|
| Default high-Z (60–80 keV)  | **CeO₂** (a = 5.4116) — broad rings, fewer overlaps, fast convergence |
| Low-Z / soft samples         | **LaB₆** (a = 4.1569) — narrower rings, better Q-resolution |
| Silicon-standard reference   | **Si** (a = 5.4310) — exquisite peak shape; matches NIST SRM 640 |
| Heavy multi-ring environments | **Al₂O₃** (trigonal) — more rings per 2θ window; less overlap than CeO₂ in some setups |